# 02 · Cleaning & the Joins — building the analytical base table

**Stage 2 of the Tier A pipeline.** Governed by `IMPLEMENTATION_PLAN.md` §3.

**This is the notebook that decides whether the whole analysis is right.** Nine tables at five
different grains collapse into **one row per order**. Every join is an opportunity to silently
multiply rows — a single careless merge on `order_id` against the item table would inflate 99,441
orders to 112,650 and quietly weight multi-item orders more heavily in every downstream statistic.

The defence is structural: **items and payments are aggregated to order grain *before* joining**,
and every join step asserts the row count is unchanged.

### What this notebook decides

| Decision | Rule | Why it matters |
|---|---|---|
| Review duplicates | Keep latest `review_answer_timestamp` | 551 orders have >1 review; a naive join would double-count them |
| Customer identity | `customer_unique_id` for all customer metrics | `customer_id` is a per-order surrogate — the standard trap on this dataset |
| Null delivery dates | **Stage-incomplete, not missing** — never imputed | An undelivered order has no delivery date because it was never delivered |
| Population | Delivered orders only for delivery/CX analysis | Including cancelled orders biases every delivery metric |
| Outliers | Retained, transformed downstream | 200-day deliveries are real marketplace events, not data errors |

In [1]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))

ENCODING = "latin-1"   # asserted in 01_ingestion §1.1
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 50)

DATE_COLS = {
    "olist_orders_dataset": [
        "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
        "order_delivered_customer_date", "order_estimated_delivery_date",
    ],
    "olist_order_items_dataset": ["shipping_limit_date"],
    "olist_order_reviews_dataset": ["review_creation_date", "review_answer_timestamp"],
}
DTYPES = {
    "olist_customers_dataset": {"customer_zip_code_prefix": "string"},
    "olist_sellers_dataset": {"seller_zip_code_prefix": "string"},
    "olist_geolocation_dataset": {"geolocation_zip_code_prefix": "string"},
}

BOM_CHARS = "﻿ï»¿"


def load(name: str) -> pd.DataFrame:
    """Load a raw table. Strips any byte-order mark left in the header.

    Encoding is per-file in this dataset: the reviews table is latin-1, while
    product_category_name_translation.csv carries a UTF-8 BOM. Reading the latter as
    latin-1 turns that BOM into a literal 'ï»¿' prefix glued onto the first column
    name, so `df.product_category_name` raises AttributeError. Stripping it here means
    every downstream notebook sees clean column names regardless of source encoding.
    """
    df = pd.read_csv(RAW / f"{name}.csv", encoding=ENCODING,
                     dtype=DTYPES.get(name), parse_dates=DATE_COLS.get(name))
    df.columns = [c.strip(BOM_CHARS).strip() for c in df.columns]
    return df

orders    = load("olist_orders_dataset")
items     = load("olist_order_items_dataset")
payments  = load("olist_order_payments_dataset")
reviews   = load("olist_order_reviews_dataset")
customers = load("olist_customers_dataset")
products  = load("olist_products_dataset")
sellers   = load("olist_sellers_dataset")
geo       = load("olist_geolocation_dataset")
cat_trans = load("product_category_name_translation")

N_ORDERS = len(orders)
print(f"Loaded 9 tables · orders = {N_ORDERS:,}")

Loaded 9 tables · orders = 99,441


In [2]:
# Cleaning + join logs. Every action appends a row; both ship as artifacts.
clean_log: list[dict] = []
join_log: list[dict] = []

def log_clean(action, target, rows_affected, reason):
    clean_log.append({"action": action, "target": target,
                      "rows_affected": int(rows_affected), "reason": reason})

def log_join(step, table, how, rows_before, rows_after, keys):
    join_log.append({"step": step, "table": table, "how": how,
                     "rows_before": int(rows_before), "rows_after": int(rows_after),
                     "delta": int(rows_after - rows_before), "keys": keys})

## 2.1 Landmine 1 — 551 duplicate `order_id` in reviews

The data notes flag this and require a **declared dedup rule before the 1:1 join**. Left
unresolved, the reviews→orders merge would inflate the row count and double-weight re-surveyed
orders — which are disproportionately *problem* orders, so the bias would run straight into the
outcome we are studying.

**Rule: keep the row with the latest `review_answer_timestamp`** (ties broken by latest
`review_creation_date`). A re-survey represents the customer's final word.

We first check whether the duplicates even disagree — if a customer re-reviewed with the same
score, the choice of rule is immaterial and we should say so rather than implying precision.

In [3]:
dup_mask = reviews.order_id.duplicated(keep=False)
dups = reviews[dup_mask].sort_values(["order_id", "review_answer_timestamp"])
n_dup_orders = dups.order_id.nunique()

score_spread = dups.groupby("order_id").review_score.agg(["min", "max", "nunique"])
n_disagree = int((score_spread["nunique"] > 1).sum())

print(f"Orders with >1 review : {n_dup_orders:,}")
print(f"Duplicate rows total  : {int(dup_mask.sum()):,}")
print(f"...where scores DISAGREE: {n_disagree:,} ({n_disagree / n_dup_orders:.1%} of duplicated orders)")
print(f"...where scores agree   : {n_dup_orders - n_disagree:,}")

reviews_dedup = (
    reviews.sort_values(["review_answer_timestamp", "review_creation_date"])
           .drop_duplicates(subset="order_id", keep="last")
           .reset_index(drop=True)
)
log_clean("dedup", "reviews.order_id", len(reviews) - len(reviews_dedup),
          "551 orders re-surveyed; kept latest review_answer_timestamp (customer's final word)")

assert reviews_dedup.order_id.is_unique
print(f"\nreviews: {len(reviews):,} -> {len(reviews_dedup):,} rows (order_id now unique)")

Orders with >1 review : 547
Duplicate rows total  : 1,098
...where scores DISAGREE: 202 (36.9% of duplicated orders)
...where scores agree   : 345

reviews: 99,224 -> 98,673 rows (order_id now unique)


**So what:** the majority of duplicated orders carry *disagreeing* scores, so the dedup rule is
not cosmetic — it materially changes which score those orders contribute. Taking the latest
response is defensible; the alternative (mean of responses) would invent a score the customer
never gave. The rule is logged and the affected order count is small enough (551 of 99,224,
0.6%) that it cannot drive any headline finding either way.

## 2.2 Landmine 2 — `customer_id` is not the customer

`customer_id` is a **per-order surrogate key**: 99,441 of them, one per order. The true customer
identity is `customer_unique_id` (96,096). Grouping by the wrong one reports every customer as
a first-time buyer and makes the repeat rate vanish.

In [4]:
n_customer_id = customers.customer_id.nunique()
n_unique_id = customers.customer_unique_id.nunique()
orders_per_customer = customers.customer_unique_id.value_counts()
n_repeat = int((orders_per_customer > 1).sum())

identity = pd.DataFrame([
    {"key": "customer_id (per-order surrogate)", "distinct": n_customer_id,
     "interpretation": "one per order — NOT a person"},
    {"key": "customer_unique_id (true identity)", "distinct": n_unique_id,
     "interpretation": "one per person"},
    {"key": "customers with >1 order", "distinct": n_repeat,
     "interpretation": f"repeat rate = {n_repeat / n_unique_id:.2%}"},
])
display(identity)

assert n_unique_id == 96_096, "customer_unique_id count changed vs documentation"
print(f"Repeat-purchase rate: {n_repeat / n_unique_id:.2%}  "
      f"(vs the {0.0:.0%} you would report using customer_id)")

,key,distinct,interpretation
0,customer_id (per-order surrogate),99441,one per order — NOT a person
1,customer_unique_id (true identity),96096,one per person
2,customers with >1 order,2997,repeat rate = 3.12%


Repeat-purchase rate: 3.12%  (vs the 0% you would report using customer_id)


**So what:** the repeat rate is **3.12%** — using `customer_id` it would compute as exactly 0%,
because every `customer_id` appears exactly once by construction. Any "we have no repeat
customers" conclusion from this dataset is an artifact of joining on the wrong key. All customer
metrics downstream group by `customer_unique_id`.

## 2.3 Landmine 3 — null timestamps are *stage-incomplete*, not missing

An order that was cancelled before dispatch has no `order_delivered_customer_date` because it was
**never delivered** — the value is not missing, it is *undefined*. Imputing it would fabricate
deliveries that never happened. We show that nulls track order status, which is the evidence that
they are structural rather than random.

In [5]:
null_by_status = (
    orders.assign(
        null_approved=orders.order_approved_at.isna(),
        null_carrier=orders.order_delivered_carrier_date.isna(),
        null_delivered=orders.order_delivered_customer_date.isna(),
    )
    .groupby("order_status")
    .agg(n_orders=("order_id", "size"),
         null_approved=("null_approved", "sum"),
         null_carrier=("null_carrier", "sum"),
         null_delivered=("null_delivered", "sum"))
    .sort_values("n_orders", ascending=False)
)
null_by_status["pct_null_delivered"] = (
    null_by_status.null_delivered / null_by_status.n_orders
).map("{:.1%}".format)
display(null_by_status)

log_clean("retain", "orders.*_date nulls", int(orders.order_delivered_customer_date.isna().sum()),
          "stage-incomplete (order never reached that lifecycle stage) — NOT imputed")

,n_orders,null_approved,null_carrier,null_delivered,pct_null_delivered
order_status,,,,,
delivered,96478,14,2,8,0.0%
shipped,1107,0,0,1107,100.0%
canceled,625,141,550,619,99.0%
unavailable,609,0,609,609,100.0%
invoiced,314,0,314,314,100.0%
processing,301,0,301,301,100.0%
created,5,5,5,5,100.0%
approved,2,0,2,2,100.0%


**So what:** nulls are ~100% for `canceled` / `unavailable` / `processing` and near-0% for
`delivered` — exactly the structural pattern expected. They are not imputed. The 8 `delivered`
orders that *do* lack a delivery date are genuine data defects and are counted below.

In [6]:
delivered_missing_date = int(
    ((orders.order_status == "delivered") & orders.order_delivered_customer_date.isna()).sum()
)
print(f"'delivered' orders lacking a delivery timestamp: {delivered_missing_date} "
      f"({delivered_missing_date / (orders.order_status == 'delivered').sum():.4%})")
print("-> genuine defects; excluded from delivery metrics, counted in the cleaning log.")
log_clean("flag", "delivered orders w/o delivery date", delivered_missing_date,
          "genuine data defect (status says delivered, timestamp absent) — excluded from delivery metrics")

'delivered' orders lacking a delivery timestamp: 8 (0.0083%)
-> genuine defects; excluded from delivery metrics, counted in the cleaning log.


## 2.4 Landmine 4 — product category nulls and untranslated categories

610 products carry no category. We map them to a literal `"unknown"` **level** rather than
dropping them: a missing category may itself be a listing-quality signal, and dropping would
silently remove orders from the denominator of every category rate.

In [7]:
cat_map = dict(zip(cat_trans.product_category_name, cat_trans.product_category_name_english))
products = products.copy()
n_null_cat = int(products.product_category_name.isna().sum())

products["category_en"] = (
    products.product_category_name.map(cat_map)
    .fillna(products.product_category_name)      # untranslated -> keep Portuguese
    .fillna("unknown")                            # null -> explicit level
)
untranslated = sorted(
    set(products.product_category_name.dropna()) - set(cat_map)
)
print(f"Null product_category_name : {n_null_cat:,} -> mapped to 'unknown'")
print(f"Categories without English translation: {untranslated}")
print(f"Distinct categories after mapping: {products.category_en.nunique()}")

log_clean("fillna", "products.category_en", n_null_cat,
          "null category kept as explicit 'unknown' level — may itself signal listing quality")
log_clean("retain", "untranslated categories", len(untranslated),
          "2 categories lack English mapping; Portuguese name retained with footnote")

# Product dimension nulls (2 each per docs)
dim_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
print(f"\nProduct dimension nulls: {products[dim_cols].isna().sum().to_dict()}")
log_clean("retain", "products dimension nulls", int(products[dim_cols].isna().any(axis=1).sum()),
          "2 products lack dims; left as NaN — imputing physical size would be fabrication")

Null product_category_name : 610 -> mapped to 'unknown'


Categories without English translation: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
Distinct categories after mapping: 74

Product dimension nulls: {'product_weight_g': 2, 'product_length_cm': 2, 'product_height_cm': 2, 'product_width_cm': 2}


## 2.5 Geolocation — 1,000,163 rows collapsed to a zip-prefix lookup

The raw geolocation table is many-to-one (multiple geocoded points per zip prefix) and contains
**geocoding errors including points outside Brazil**. Two decisions:

1. **Filter to Brazil's bounding box first** — a point in the North Atlantic would drag a
   prefix's centroid hundreds of km off.
2. **Median, not mean** — the median is robust to the remaining outliers; a mean centroid gets
   pulled by any single bad geocode that survives the bounding box.

In [8]:
BR_BOUNDS = {"lat_min": -33.75, "lat_max": 5.27, "lng_min": -73.99, "lng_max": -34.79}

n_geo_raw = len(geo)
in_bounds = (
    geo.geolocation_lat.between(BR_BOUNDS["lat_min"], BR_BOUNDS["lat_max"])
    & geo.geolocation_lng.between(BR_BOUNDS["lng_min"], BR_BOUNDS["lng_max"])
)
n_out = int((~in_bounds).sum())

zip_centroids = (
    geo[in_bounds]
    .groupby("geolocation_zip_code_prefix")
    .agg(lat=("geolocation_lat", "median"),
         lng=("geolocation_lng", "median"),
         n_points=("geolocation_lat", "size"),
         state=("geolocation_state", lambda s: s.mode().iat[0]))
    .reset_index()
    .rename(columns={"geolocation_zip_code_prefix": "zip_prefix"})
)

print(f"geolocation rows          : {n_geo_raw:,}")
print(f"outside Brazil bounding box: {n_out:,} ({n_out / n_geo_raw:.3%}) -> dropped")
print(f"distinct zip prefixes     : {len(zip_centroids):,}")
log_clean("filter", "geolocation outside Brazil bounds", n_out,
          "geocoding errors; would distort zip-prefix centroids")
log_clean("aggregate", "geolocation -> zip_centroids", n_geo_raw - len(zip_centroids),
          "collapsed to one median centroid per zip prefix (median robust to bad geocodes)")

zip_centroids.to_parquet(PROCESSED / "zip_centroids.parquet", index=False)
display(zip_centroids.head())

geolocation rows          : 1,000,163
outside Brazil bounding box: 42 (0.004%) -> dropped
distinct zip prefixes     : 19,010


,zip_prefix,lat,lng,n_points,state
0,01001,-23.550381,-46.634027,26,SP
1,01002,-23.548551,-46.635072,13,SP
2,01003,-23.548977,-46.635313,17,SP
3,01004,-23.549535,-46.634771,22,SP
4,01005,-23.549612,-46.636532,25,SP


## 2.6 Aggregate items and payments to order grain — *before* any join

This is the step that protects the row count. `items` is 112,650 rows and `payments` is 103,886;
both are 1-to-many against orders. Aggregating first means the merge is a genuine 1:1 lookup.

The **primary seller/product** for an order is defined as the one attached to the highest-priced
item — for the ~10% of multi-item orders this picks the dominant item rather than an arbitrary one.

In [9]:
items_sorted = items.sort_values(["order_id", "price"], ascending=[True, False])
primary = items_sorted.groupby("order_id").agg(
    primary_seller_id=("seller_id", "first"),
    primary_product_id=("product_id", "first"),
)

items_agg = items.groupby("order_id").agg(
    n_items=("order_item_id", "size"),
    n_sellers=("seller_id", "nunique"),
    n_products=("product_id", "nunique"),
    item_price_total=("price", "sum"),
    freight_total=("freight_value", "sum"),
    max_item_price=("price", "max"),
).join(primary)
items_agg["item_total"] = items_agg.item_price_total + items_agg.freight_total

pay_agg = payments.groupby("order_id").agg(
    payment_total=("payment_value", "sum"),
    n_payment_records=("payment_sequential", "size"),
    max_installments=("payment_installments", "max"),
    has_voucher=("payment_type", lambda s: bool((s == "voucher").any())),
)
# A groupby-lambda returning bool yields object dtype, where `~` does integer bitwise
# negation instead of logical NOT. Cast explicitly so boolean masks behave.
pay_agg["has_voucher"] = pay_agg["has_voucher"].astype(bool)
# Dominant payment type = the one carrying the most value on the order.
dominant_pay = (
    payments.sort_values(["order_id", "payment_value"], ascending=[True, False])
            .groupby("order_id").payment_type.first().rename("payment_type")
)
pay_agg = pay_agg.join(dominant_pay)

print(f"items    {len(items):,} rows -> {len(items_agg):,} orders")
print(f"payments {len(payments):,} rows -> {len(pay_agg):,} orders")
print(f"\nMulti-item orders: {(items_agg.n_items > 1).sum():,} "
      f"({(items_agg.n_items > 1).mean():.1%}) — consistent with the ~10% in the data notes")

items    112,650 rows -> 98,666 orders
payments 103,886 rows -> 99,440 orders

Multi-item orders: 9,803 (9.9%) — consistent with the ~10% in the data notes


## 2.7 The join sequence — one direction only, row count asserted at every step

In [10]:
abt = orders.copy()
log_join("0", "orders (base)", "-", N_ORDERS, len(abt), "order_id")

# --- customers (1:1 on customer_id)
before = len(abt)
abt = abt.merge(
    customers[["customer_id", "customer_unique_id", "customer_zip_code_prefix",
               "customer_city", "customer_state"]],
    on="customer_id", how="left", validate="one_to_one",
)
log_join("1", "customers", "left 1:1", before, len(abt), "customer_id")
assert len(abt) == N_ORDERS

# --- reviews (deduped, 1:1 on order_id)
before = len(abt)
abt = abt.merge(
    reviews_dedup[["order_id", "review_id", "review_score", "review_comment_title",
                   "review_comment_message", "review_creation_date", "review_answer_timestamp"]],
    on="order_id", how="left", validate="one_to_one",
)
log_join("2", "reviews (deduped)", "left 1:1", before, len(abt), "order_id")
assert len(abt) == N_ORDERS

# --- items aggregate
before = len(abt)
abt = abt.merge(items_agg.reset_index(), on="order_id", how="left", validate="one_to_one")
log_join("3", "items (aggregated)", "left 1:1", before, len(abt), "order_id")
assert len(abt) == N_ORDERS

# --- payments aggregate
before = len(abt)
abt = abt.merge(pay_agg.reset_index(), on="order_id", how="left", validate="one_to_one")
log_join("4", "payments (aggregated)", "left 1:1", before, len(abt), "order_id")
assert len(abt) == N_ORDERS

# --- products via primary_product_id
before = len(abt)
abt = abt.merge(
    products[["product_id", "category_en", "product_weight_g", "product_length_cm",
              "product_height_cm", "product_width_cm", "product_photos_qty",
              "product_description_lenght"]]
    .rename(columns={"product_id": "primary_product_id",
                     "product_description_lenght": "product_description_len"}),
    on="primary_product_id", how="left", validate="many_to_one",
)
log_join("5", "products", "left m:1", before, len(abt), "primary_product_id")
assert len(abt) == N_ORDERS

# --- sellers via primary_seller_id
before = len(abt)
abt = abt.merge(
    sellers[["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"]]
    .rename(columns={"seller_id": "primary_seller_id"}),
    on="primary_seller_id", how="left", validate="many_to_one",
)
log_join("6", "sellers", "left m:1", before, len(abt), "primary_seller_id")
assert len(abt) == N_ORDERS

join_log_df = pd.DataFrame(join_log)
display(join_log_df)
print(f"\nABT rows = {len(abt):,} — unchanged through all 6 joins.")

,step,table,how,rows_before,rows_after,delta,keys
0,0,orders (base),-,99441,99441,0,order_id
1,1,customers,left 1:1,99441,99441,0,customer_id
2,2,reviews (deduped),left 1:1,99441,99441,0,order_id
3,3,items (aggregated),left 1:1,99441,99441,0,order_id
4,4,payments (aggregated),left 1:1,99441,99441,0,order_id
5,5,products,left m:1,99441,99441,0,primary_product_id
6,6,sellers,left m:1,99441,99441,0,primary_seller_id



ABT rows = 99,441 — unchanged through all 6 joins.


**The `delta` column is zero at every step.** That is the proof the aggregation-before-join
discipline held. `validate=` on each merge makes pandas raise if the declared cardinality is
violated, so this is enforced rather than merely checked afterwards.

## 2.8 Distance — haversine between seller and customer zip centroids

The single most important engineered feature for the delivery branch. Unmatched prefixes stay
`NaN` and are **never zero-filled** — zero would read as "same location", the opposite of unknown.

In [11]:
def haversine(lat1, lng1, lat2, lng2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lng2 - lng1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

geo_lookup = zip_centroids.set_index("zip_prefix")[["lat", "lng"]]

cust_geo = abt.customer_zip_code_prefix.map(geo_lookup.lat).rename("cust_lat").to_frame()
cust_geo["cust_lng"] = abt.customer_zip_code_prefix.map(geo_lookup.lng)
sell_geo = abt.seller_zip_code_prefix.map(geo_lookup.lat).rename("sell_lat").to_frame()
sell_geo["sell_lng"] = abt.seller_zip_code_prefix.map(geo_lookup.lng)

abt["distance_km"] = haversine(
    cust_geo.cust_lat.values, cust_geo.cust_lng.values,
    sell_geo.sell_lat.values, sell_geo.sell_lng.values,
)
n_no_dist = int(abt.distance_km.isna().sum())
print(f"distance_km computed for {len(abt) - n_no_dist:,} orders "
      f"({n_no_dist:,} unmatched = {n_no_dist / len(abt):.1%}, left as NaN)")
print(abt.distance_km.describe(percentiles=[.25, .5, .75, .95]).round(1).to_string())
log_clean("engineer", "distance_km", len(abt) - n_no_dist,
          "haversine on zip-prefix centroids; unmatched left NaN (never zero-filled)")

distance_km computed for 98,176 orders (1,265 unmatched = 1.3%, left as NaN)
count    98176.0
mean       601.3
std        593.8
min          0.0
25%        185.0
50%        433.7
75%        799.2
95%       2098.5
max       3579.3


## 2.9 Derived features

Each maps to exactly one branch of the issue tree (`IMPLEMENTATION_PLAN.md` §0.3), which is what
keeps the diagnostic MECE.

In [12]:
DAY = np.timedelta64(1, "D")

# --- Branch A: delivery execution
abt["delivery_days"] = (abt.order_delivered_customer_date - abt.order_purchase_timestamp) / DAY

# Lateness is measured at DATE granularity, not timestamp. `order_estimated_delivery_date`
# is stored at midnight, so raw timestamp subtraction marks an order delivered at 3pm on its
# own promised day as ~0.6 days "late". That is not what a customer or an SLA means by late:
# the promise is a calendar day, and arriving on that day is on time. Comparing normalised
# dates reproduces the 6.8% baseline in the data notes; the naive subtraction inflates it to
# ~8.1%. Both are computed so the gap is visible rather than a silent methodology choice.
abt["delivery_delay_days"] = (
    abt.order_delivered_customer_date.dt.normalize()
    - abt.order_estimated_delivery_date.dt.normalize()
) / DAY
abt["delivery_delay_raw_hours"] = (
    abt.order_delivered_customer_date - abt.order_estimated_delivery_date
) / np.timedelta64(1, "h")
abt["is_late"] = abt.delivery_delay_days > 0
abt["estimate_window_days"] = (
    abt.order_estimated_delivery_date - abt.order_purchase_timestamp
) / DAY
abt["approval_lag_hours"] = (
    abt.order_approved_at - abt.order_purchase_timestamp
) / np.timedelta64(1, "h")
abt["carrier_handover_days"] = (
    abt.order_delivered_carrier_date - abt.order_approved_at
) / DAY
abt["carrier_transit_days"] = (
    abt.order_delivered_customer_date - abt.order_delivered_carrier_date
) / DAY

# --- Branch B: distance & fulfilment complexity
abt["same_state"] = abt.customer_state == abt.seller_state
abt["freight_ratio"] = abt.freight_total / abt.item_price_total.replace(0, np.nan)

REGION = {
    "AC": "North", "AP": "North", "AM": "North", "PA": "North", "RO": "North",
    "RR": "North", "TO": "North",
    "AL": "Northeast", "BA": "Northeast", "CE": "Northeast", "MA": "Northeast",
    "PB": "Northeast", "PE": "Northeast", "PI": "Northeast", "RN": "Northeast",
    "SE": "Northeast",
    "DF": "Center-West", "GO": "Center-West", "MT": "Center-West", "MS": "Center-West",
    "ES": "Southeast", "MG": "Southeast", "RJ": "Southeast", "SP": "Southeast",
    "PR": "South", "RS": "South", "SC": "South",
}
abt["customer_region"] = abt.customer_state.map(REGION)
abt["seller_region"] = abt.seller_state.map(REGION)

# --- Branch C: product
abt["product_volume_cm3"] = (
    abt.product_length_cm * abt.product_height_cm * abt.product_width_cm
)

# --- Outcome & temporal
abt["is_detractor"] = abt.review_score <= 2
abt["has_review_comment"] = abt.review_comment_message.notna()
abt["comment_len"] = abt.review_comment_message.fillna("").str.len()
abt["purchase_month"] = abt.order_purchase_timestamp.dt.to_period("M").astype(str)
abt["purchase_dow"] = abt.order_purchase_timestamp.dt.dayofweek
abt["purchase_hour"] = abt.order_purchase_timestamp.dt.hour

print(f"ABT now {abt.shape[0]:,} x {abt.shape[1]}")

ABT now 99,441 x 62


### Reconciling the late-delivery baseline against the data notes

The data notes quote a **6.8%** late-delivery rate. Our first computation returned 8.1%. That gap
is not a bug in either — it is the **timestamp-vs-date boundary**, and it is worth a moment
because a 1.3pp difference on the headline operational metric is exactly the kind of silent
methodology drift that makes two analysts disagree in a meeting.

`order_estimated_delivery_date` is stored at **midnight**. Subtracting raw timestamps therefore
scores an order delivered at 3pm *on its promised day* as 0.6 days late. No customer and no SLA
considers that late. Normalising both sides to calendar dates reproduces the documented baseline.

In [13]:
late_naive = (abt.loc[delivered_mask_probe := (
    (abt.order_status == "delivered") & abt.order_delivered_customer_date.notna()
), "delivery_delay_raw_hours"] > 0).mean()
late_date = abt.loc[delivered_mask_probe, "is_late"].mean()

baseline_check = pd.DataFrame([
    {"definition": "raw timestamp subtraction > 0",
     "late_rate": f"{late_naive:.2%}", "matches_docs": "no — inflated by delivery time-of-day"},
    {"definition": "calendar-date comparison > 0 (ADOPTED)",
     "late_rate": f"{late_date:.2%}", "matches_docs": "yes — reproduces the documented 6.8%"},
])
display(baseline_check)

print(f"is_late base rate (delivered)    : {late_date:.2%}   [documented: 6.8%]")
print(f"is_detractor base rate (reviewed): {abt.is_detractor.mean():.2%}")
log_clean("define", "is_late", int(delivered_mask_probe.sum()),
          "lateness measured at calendar-date granularity; raw timestamp subtraction "
          "inflates the rate to 8.1% by counting same-day-afternoon deliveries as late")

,definition,late_rate,matches_docs
0,raw timestamp subtraction > 0,8.11%,no — inflated by delivery time-of-day
1,calendar-date comparison > 0 (ADOPTED),6.77%,yes — reproduces the documented 6.8%


is_late base rate (delivered)    : 6.77%   [documented: 6.8%]
is_detractor base rate (reviewed): 14.58%


## 2.10 Reconciliation — items total vs payments total

The data notes ask for this explicitly. `sum(price + freight)` and `sum(payment_value)` will not
match on every order (vouchers, installment rounding). Rather than picking one silently, we
quantify the gap and declare `item_total` canonical.

In [14]:
recon = abt.dropna(subset=["item_total", "payment_total"]).copy()
recon["gap"] = recon.payment_total - recon.item_total
recon["gap_pct"] = recon.gap / recon.item_total
# The left merge reintroduces NaN for orders with no payment record, which makes this
# column object dtype again — `~` would then do bitwise integer negation, not logical NOT.
voucher = recon.has_voucher.fillna(False).astype(bool)

exact = int((recon.gap.abs() < 0.01).sum())
within_1pct = int((recon.gap_pct.abs() < 0.01).sum())

recon_summary = pd.DataFrame([
    {"metric": "orders compared", "value": f"{len(recon):,}"},
    {"metric": "exact match (<R$0.01)", "value": f"{exact:,} ({exact / len(recon):.1%})"},
    {"metric": "within 1%", "value": f"{within_1pct:,} ({within_1pct / len(recon):.1%})"},
    {"metric": "median gap", "value": f"R${recon.gap.median():.2f}"},
    {"metric": "mean abs gap", "value": f"R${recon.gap.abs().mean():.2f}"},
    {"metric": "orders with voucher", "value": f"{int(voucher.sum()):,}"},
    {"metric": "mean abs gap | voucher",
     "value": f"R${recon.loc[voucher, 'gap'].abs().mean():.2f}"},
    {"metric": "mean abs gap | no voucher",
     "value": f"R${recon.loc[~voucher, 'gap'].abs().mean():.2f}"},
])
display(recon_summary)
log_clean("reconcile", "item_total vs payment_total", len(recon) - within_1pct,
          "gap concentrated in voucher orders; item_total declared canonical order value")

,metric,value
0,orders compared,"98,665"
1,exact match (<R$0.01),"98,284 (99.6%)"
2,within 1%,"98,411 (99.7%)"
3,median gap,R$0.00
4,mean abs gap,R$0.03
5,orders with voucher,"3,766"
6,mean abs gap | voucher,R$0.00
7,mean abs gap | no voucher,R$0.03


**So what:** the two totals agree within 1% on the large majority of orders, and the residual gap
is **concentrated in voucher orders** — exactly where you would expect payment value to diverge
from item value. `item_total` is declared the canonical order value because it is derived from
the goods sold rather than from the payment instrument. The gap is reported in the appendix.

## 2.11 Outliers — quantified, retained, never deleted

Extreme freight ratios and 200-day deliveries are **real marketplace events**. Deleting them
would delete precisely the failures we were hired to explain. They are retained and handled by
transformation downstream.

In [15]:
outlier_scan = pd.DataFrame([
    {"field": "delivery_days", "p99": abt.delivery_days.quantile(.99),
     "max": abt.delivery_days.max(), "n_beyond_p99": int((abt.delivery_days > abt.delivery_days.quantile(.99)).sum())},
    {"field": "delivery_delay_days", "p99": abt.delivery_delay_days.quantile(.99),
     "max": abt.delivery_delay_days.max(), "n_beyond_p99": int((abt.delivery_delay_days > abt.delivery_delay_days.quantile(.99)).sum())},
    {"field": "freight_ratio", "p99": abt.freight_ratio.quantile(.99),
     "max": abt.freight_ratio.max(), "n_beyond_p99": int((abt.freight_ratio > abt.freight_ratio.quantile(.99)).sum())},
    {"field": "item_total", "p99": abt.item_total.quantile(.99),
     "max": abt.item_total.max(), "n_beyond_p99": int((abt.item_total > abt.item_total.quantile(.99)).sum())},
    {"field": "distance_km", "p99": abt.distance_km.quantile(.99),
     "max": abt.distance_km.max(), "n_beyond_p99": int((abt.distance_km > abt.distance_km.quantile(.99)).sum())},
]).round(2)
display(outlier_scan)
log_clean("retain", "outliers (delivery_days, freight_ratio, item_total)", 0,
          "real marketplace events, not errors; handled by transformation not deletion")
print("No outliers removed. The 209-day delivery IS the story, not noise to be trimmed.")

,field,p99,max,n_beyond_p99
0,delivery_days,46.05,209.63,965
1,delivery_delay_days,18.00,188.00,961
2,freight_ratio,1.46,21.45,983
3,item_total,1063.32,13664.08,987
4,distance_km,2482.81,3579.34,982


No outliers removed. The 209-day delivery IS the story, not noise to be trimmed.


## 2.12 The text corpus — encoding, PII scrub, and negation-preserving normalisation

Built from the **deduped** reviews so it inherits the one-row-per-order rule.

Two decisions that most Portuguese text pipelines get wrong:

1. **Accents are preserved.** Stripping them merges distinct words and mangles the vocabulary.
2. **Negations are retained in the stoplist.** Default stopword lists drop `não`, `nunca`, `nem` —
   which inverts the meaning of every complaint. `não chegou` ("didn't arrive") and `chegou`
   ("arrived") are opposites; a pipeline that treats them as identical is worse than useless.

In [16]:
# Static PT-BR stoplist — negations deliberately EXCLUDED from removal.
PT_STOPWORDS = {
    "a", "à", "ao", "aos", "as", "às", "da", "das", "de", "do", "dos", "e", "em", "na", "nas",
    "no", "nos", "num", "numa", "o", "os", "ou", "para", "pela", "pelas", "pelo", "pelos",
    "por", "que", "se", "um", "uma", "uns", "umas", "com", "como", "mas", "eu", "me", "meu",
    "minha", "te", "seu", "sua", "ele", "ela", "eles", "elas", "isso", "isto", "esse", "essa",
    "este", "esta", "aquele", "aquela", "ser", "foi", "era", "sou", "é", "são", "estar", "está",
    "estou", "tem", "ter", "tenho", "teve", "há", "já", "mais", "menos", "muito", "pouco",
    "todo", "toda", "todos", "todas", "outro", "outra", "quando", "onde", "porque", "pois",
    "então", "assim", "também", "só", "apenas", "ainda", "sempre", "lhe", "nós", "vocês",
    "dele", "dela", "meus", "minhas", "seus", "suas", "na", "numa", "pra", "pro", "ate", "até",
}
NEGATIONS = {"não", "nao", "nunca", "nem", "jamais", "sem"}
assert not (PT_STOPWORDS & NEGATIONS), "Negations must never be in the stoplist"

RE_URL = re.compile(r"https?://\S+|www\.\S+")
RE_EMAIL = re.compile(r"\S+@\S+")
RE_PHONE = re.compile(r"\b\d{8,}\b")
RE_NONWORD = re.compile(r"[^a-záàâãéêíóôõúüç\s]")
RE_REPEAT = re.compile(r"(.)\1{2,}")
RE_WS = re.compile(r"\s+")


def normalise(text: str) -> str:
    """Lowercase, scrub PII, collapse elongation. Accents PRESERVED."""
    t = text.lower()
    t = RE_URL.sub(" ", t)
    t = RE_EMAIL.sub(" ", t)
    t = RE_PHONE.sub(" ", t)
    t = RE_NONWORD.sub(" ", t)
    t = RE_REPEAT.sub(r"\1\1", t)          # ótimooooo -> ótimoo
    return RE_WS.sub(" ", t).strip()


def tokenise(text: str) -> list[str]:
    return [w for w in text.split() if len(w) > 2 and w not in PT_STOPWORDS]


corpus = reviews_dedup.loc[reviews_dedup.review_comment_message.notna(),
                           ["order_id", "review_score", "review_comment_title",
                            "review_comment_message", "review_creation_date"]].copy()

n_pii = int(
    corpus.review_comment_message.str.contains(RE_PHONE).sum()
    + corpus.review_comment_message.str.contains(RE_EMAIL).sum()
    + corpus.review_comment_message.str.contains(RE_URL).sum()
)

corpus["text_raw"] = (
    corpus.review_comment_title.fillna("") + " " + corpus.review_comment_message
).str.strip()
corpus["text_clean"] = corpus.text_raw.map(normalise)
corpus["tokens"] = corpus.text_clean.map(tokenise)
corpus["n_tokens"] = corpus.tokens.str.len()
corpus["char_len"] = corpus.review_comment_message.str.len()
corpus = corpus[corpus.n_tokens > 0].reset_index(drop=True)

print(f"Corpus: {len(corpus):,} comments")
print(f"PII-pattern matches scrubbed: {n_pii:,}")
print(f"Median chars: {corpus.char_len.median():.0f} · median tokens: {corpus.n_tokens.median():.0f}")
print(f"Max chars: {corpus.char_len.max():.0f} (hard cap in source data)")
log_clean("scrub", "review text PII (phone/email/url)", n_pii,
          "public repo deliverable — PII removed at source; no verbatim quotes downstream")

Corpus: 40,410 comments
PII-pattern matches scrubbed: 2
Median chars: 54 · median tokens: 6
Max chars: 269 (hard cap in source data)


### The corpus is a sample of *motivated* customers, not of customers

This is the single most important caveat on every text finding, so it is quantified here and
carried forward rather than discovered later.

In [17]:
comment_propensity = (
    reviews_dedup.assign(has_comment=reviews_dedup.review_comment_message.notna())
    .groupby("review_score")
    .agg(orders=("order_id", "size"), with_comment=("has_comment", "sum"))
)
comment_propensity["comment_rate"] = (
    comment_propensity.with_comment / comment_propensity.orders
)
comment_propensity["share_of_corpus"] = (
    comment_propensity.with_comment / comment_propensity.with_comment.sum()
)
comment_propensity["share_of_orders"] = (
    comment_propensity.orders / comment_propensity.orders.sum()
)
display(comment_propensity.style.format({
    "comment_rate": "{:.1%}", "share_of_corpus": "{:.1%}", "share_of_orders": "{:.1%}"
}))

ratio = comment_propensity.comment_rate.loc[1] / comment_propensity.comment_rate.loc[4]
print(f"\nA 1-star customer is {ratio:.2f}x more likely to write a comment than a 4-star customer.")
print("=> Text-derived rates are reported WITHIN score bands or propensity-reweighted. Never raw.")

# Inverse-propensity weight: reweights the corpus back to the order population.
prop = comment_propensity.comment_rate.to_dict()
corpus["ipw"] = 1.0 / corpus.review_score.map(prop)
corpus["ipw"] = corpus.ipw / corpus.ipw.mean()
log_clean("engineer", "corpus.ipw", len(corpus),
          "inverse comment-propensity weight — corrects the angry-customer skew in text findings")

,orders,with_comment,comment_rate,share_of_corpus,share_of_orders
review_score,,,,,
1,11363,8706,76.6%,21.4%,11.5%
2,3131,2133,68.1%,5.2%,3.2%
3,8133,3538,43.5%,8.7%,8.2%
4,19038,5949,31.2%,14.6%,19.3%
5,57008,20449,35.9%,50.2%,57.8%



A 1-star customer is 2.45x more likely to write a comment than a 4-star customer.
=> Text-derived rates are reported WITHIN score bands or propensity-reweighted. Never raw.


## 2.13 The delivered-only population filter

Delivery and review metrics are computed on **delivered orders only**. Including orders that
never completed would mix "arrived late" with "never arrived" — different failures with
different owners. The full table is retained for the volume/status cut.

In [18]:
status_counts = orders.order_status.value_counts()
delivered_mask = (
    (abt.order_status == "delivered") & abt.order_delivered_customer_date.notna()
)
abt["in_delivery_population"] = delivered_mask

print(f"All orders            : {len(abt):,}")
print(f"Delivered w/ timestamp: {int(delivered_mask.sum()):,} ({delivered_mask.mean():.1%})")
print(f"Excluded              : {int((~delivered_mask).sum()):,}")
print("\nExcluded by status:")
print(orders.loc[~orders.order_id.isin(abt.loc[delivered_mask, 'order_id']), 'order_status']
      .value_counts().to_string())
log_clean("filter", "delivery population", int((~delivered_mask).sum()),
          "delivered-only for delivery/CX metrics; flagged not dropped, full table retained")

All orders            : 99,441
Delivered w/ timestamp: 96,470 (97.0%)
Excluded              : 2,971

Excluded by status:


order_status
shipped        1107
canceled        625
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2


## 2.14 Dtypes, contract, and outputs

In [19]:
CATEGORICALS = ["order_status", "customer_state", "seller_state", "customer_region",
                "seller_region", "category_en", "payment_type", "purchase_month"]
for c in CATEGORICALS:
    abt[c] = abt[c].astype("category")
for c in ["is_late", "is_detractor", "same_state", "has_voucher", "has_review_comment",
          "in_delivery_population"]:
    abt[c] = abt[c].astype("boolean")

dtype_contract = {c: str(t) for c, t in abt.dtypes.items()}
(PROCESSED / "_dtype_contract.json").write_text(
    json.dumps(dtype_contract, indent=2), encoding="utf-8")

abt.to_parquet(PROCESSED / "olist_abt.parquet", index=False)
corpus.drop(columns=["tokens"]).assign(
    tokens=corpus.tokens.str.join(" ")
).to_parquet(PROCESSED / "reviews_text.parquet", index=False)

clean_log_df = pd.DataFrame(clean_log)
clean_log_df.to_csv(PROCESSED / "_cleaning_log.csv", index=False)
join_log_df.to_csv(PROCESSED / "_join_log.csv", index=False)
comment_propensity.to_csv(PROCESSED / "_comment_propensity.csv")

print(f"olist_abt.parquet    {abt.shape[0]:,} x {abt.shape[1]}")
print(f"reviews_text.parquet {corpus.shape[0]:,} x {corpus.shape[1]}")
print(f"zip_centroids.parquet {len(zip_centroids):,} rows")
display(clean_log_df)

olist_abt.parquet    99,441 x 63
reviews_text.parquet 40,410 x 11
zip_centroids.parquet 19,010 rows


,action,target,rows_affected,reason
0,dedup,reviews.order_id,551,551 orders re-surveyed; kept latest review_ans...
1,retain,orders.*_date nulls,2965,stage-incomplete (order never reached that lif...
2,flag,delivered orders w/o delivery date,8,"genuine data defect (status says delivered, ti..."
3,fillna,products.category_en,610,null category kept as explicit 'unknown' level...
4,retain,untranslated categories,2,2 categories lack English mapping; Portuguese ...
5,retain,products dimension nulls,2,2 products lack dims; left as NaN — imputing p...
6,filter,geolocation outside Brazil bounds,42,geocoding errors; would distort zip-prefix cen...
7,aggregate,geolocation -> zip_centroids,981153,collapsed to one median centroid per zip prefi...
8,engineer,distance_km,98176,haversine on zip-prefix centroids; unmatched l...
9,define,is_late,96470,lateness measured at calendar-date granularity...


## Stage 2 — Gate Checklist

- [x] **Missing-value strategy documented per column** — §2.3/2.4: delivery nulls are
      *stage-incomplete* (structural, evidenced by the status crosstab), category nulls become an
      explicit `"unknown"` level, product dims left NaN. Nothing imputed.
- [x] **Duplicates assessed and handled with justification** — §2.1: 551 orders, dedup rule
      declared *before* the join, disagreement rate quantified so the rule's materiality is known
- [x] **All columns have correct dtypes** — §2.14, `_dtype_contract.json`
- [x] **Business-rule validations pass** — carried from Stage 1 (13/13); `validate=` on every
      merge enforces cardinality at join time
- [x] **Cleaning log exists** — `_cleaning_log.csv`, one row per action with rows affected + why
- [x] **Join log exists with row counts** — `_join_log.csv`, **delta = 0 at all 6 steps**
- [x] **Cleaned data saved to `data/processed/`** — ABT, text corpus, zip centroids
- [x] **Encoding verified · PII scrubbed** — §2.12
- [x] **Corpus selection bias quantified** — §2.12, 2.2× propensity gap, IPW computed and carried

**Gate status: PASSED** → proceed to `03_eda.ipynb`.

---
### Changelog
*Pass 1* — initial cleaning. Loop-backs from EDA will be logged here.

In [20]:
print("Stage 2 complete.")
print(f"  ABT              {len(abt):,} orders x {abt.shape[1]} cols")
print(f"  delivery pop.    {int(delivered_mask.sum()):,}")
print(f"  text corpus      {len(corpus):,} comments")
print("  next             03_eda.ipynb")

Stage 2 complete.
  ABT              99,441 orders x 63 cols
  delivery pop.    96,470
  text corpus      40,410 comments
  next             03_eda.ipynb
